# Cross-Lingual Idiom Entity Difference Analysis

This notebook analyzes how Chinese and English idioms with similar figurative meanings use different entities.

**Goal**: Understand cultural differences in how concepts are expressed through different entities (animals, objects, body parts, etc.)

In [ ]:
import json
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

# Try to import seaborn, but make it optional
try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False
    print("Note: seaborn not installed. Using matplotlib defaults.")

# Set style
if HAS_SEABORN:
    plt.style.use('seaborn-v0_8-whitegrid')
    sns.set_palette("husl")
else:
    plt.style.use('ggplot')

# For Chinese characters
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 1. Load Data

In [ ]:
# Load cross-lingual pairs
pairs = []
with open('/home/jiaruil5/culture_pretrain/CultureInFigurativeLanguage/culture/data/idioms/cross_lingual_pairs.jsonl', 'r') as f:
    for line in f:
        if line.strip():
            pairs.append(json.loads(line))

print(f"Total pairs loaded: {len(pairs)}")

# Convert to DataFrame for easier analysis
df = pd.DataFrame(pairs)
df.head()

## 2. Basic Statistics

In [ ]:
# Count entity presence
df['zh_has_entities'] = df['zh_entities'].apply(lambda x: len(x) > 0)
df['en_has_entities'] = df['en_entities'].apply(lambda x: len(x) > 0)
df['both_have_entities'] = df['zh_has_entities'] & df['en_has_entities']
df['zh_entity_count'] = df['zh_entities'].apply(len)
df['en_entity_count'] = df['en_entities'].apply(len)

print("=" * 60)
print("ENTITY PRESENCE STATISTICS")
print("=" * 60)
print(f"\nTotal pairs: {len(df)}")
print(f"Pairs where Chinese has entities: {df['zh_has_entities'].sum()} ({df['zh_has_entities'].mean()*100:.1f}%)")
print(f"Pairs where English has entities: {df['en_has_entities'].sum()} ({df['en_has_entities'].mean()*100:.1f}%)")
print(f"Pairs where BOTH have entities: {df['both_have_entities'].sum()} ({df['both_have_entities'].mean()*100:.1f}%)")
print(f"\nAverage Chinese entities per idiom: {df['zh_entity_count'].mean():.2f}")
print(f"Average English entities per idiom: {df['en_entity_count'].mean():.2f}")

In [ ]:
# Similarity distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall similarity distribution
axes[0].hist(df['similarity'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Cosine Similarity')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Cross-Lingual Similarity Scores')
axes[0].axvline(df['similarity'].mean(), color='red', linestyle='--', label=f'Mean: {df["similarity"].mean():.3f}')
axes[0].legend()

# Similarity by entity presence
both_entities = df[df['both_have_entities']]['similarity']
not_both = df[~df['both_have_entities']]['similarity']
axes[1].hist([both_entities, not_both], bins=30, label=['Both have entities', 'Not both'], alpha=0.7)
axes[1].set_xlabel('Cosine Similarity')
axes[1].set_ylabel('Count')
axes[1].set_title('Similarity by Entity Presence')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Entity Frequency Analysis

In [ ]:
# Count all entities
zh_entity_counter = Counter()
en_entity_counter = Counter()

for _, row in df.iterrows():
    zh_entity_counter.update(row['zh_entities'])
    en_entity_counter.update(row['en_entities'])

print("=" * 60)
print("TOP 30 CHINESE ENTITIES")
print("=" * 60)
for entity, count in zh_entity_counter.most_common(30):
    print(f"  {entity}: {count}")

print("\n" + "=" * 60)
print("TOP 30 ENGLISH ENTITIES")
print("=" * 60)
for entity, count in en_entity_counter.most_common(30):
    print(f"  {entity}: {count}")

In [ ]:
# Visualize top entities
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top Chinese entities
top_zh = zh_entity_counter.most_common(20)
zh_entities_list, zh_counts = zip(*top_zh) if top_zh else ([], [])
axes[0].barh(range(len(zh_entities_list)), zh_counts, color='coral')
axes[0].set_yticks(range(len(zh_entities_list)))
axes[0].set_yticklabels(zh_entities_list)
axes[0].invert_yaxis()
axes[0].set_xlabel('Frequency')
axes[0].set_title('Top 20 Chinese Entities')

# Top English entities
top_en = en_entity_counter.most_common(20)
en_entities_list, en_counts = zip(*top_en) if top_en else ([], [])
axes[1].barh(range(len(en_entities_list)), en_counts, color='steelblue')
axes[1].set_yticks(range(len(en_entities_list)))
axes[1].set_yticklabels(en_entities_list)
axes[1].invert_yaxis()
axes[1].set_xlabel('Frequency')
axes[1].set_title('Top 20 English Entities')

plt.tight_layout()
plt.show()

## 4. Entity Co-occurrence Analysis

When a Chinese idiom uses entity X, what English entities tend to appear in semantically similar English idioms?

In [ ]:
# Build co-occurrence matrix for pairs where both have entities
both_df = df[df['both_have_entities']].copy()
print(f"Analyzing {len(both_df)} pairs where both languages have entities")

# Entity pair co-occurrences
entity_pairs = Counter()
zh_to_en_mapping = defaultdict(Counter)
en_to_zh_mapping = defaultdict(Counter)

for _, row in both_df.iterrows():
    for zh_e in row['zh_entities']:
        for en_e in row['en_entities']:
            entity_pairs[(zh_e, en_e)] += 1
            zh_to_en_mapping[zh_e][en_e] += 1
            en_to_zh_mapping[en_e][zh_e] += 1

In [ ]:
# Most common entity pairs
print("=" * 60)
print("TOP 30 CROSS-LINGUAL ENTITY PAIRS (ZH -> EN)")
print("=" * 60)
for (zh_e, en_e), count in entity_pairs.most_common(30):
    print(f"  {zh_e} <-> {en_e}: {count}")

In [ ]:
# For top Chinese entities, show what English entities they map to
print("=" * 60)
print("CHINESE ENTITY -> ENGLISH ENTITY MAPPINGS")
print("=" * 60)

top_zh_entities = [e for e, _ in zh_entity_counter.most_common(15)]
for zh_e in top_zh_entities:
    if zh_e in zh_to_en_mapping:
        en_mappings = zh_to_en_mapping[zh_e].most_common(5)
        en_str = ", ".join([f"{e}({c})" for e, c in en_mappings])
        print(f"\n{zh_e} -> {en_str}")

In [ ]:
# For top English entities, show what Chinese entities they map to
print("=" * 60)
print("ENGLISH ENTITY -> CHINESE ENTITY MAPPINGS")
print("=" * 60)

top_en_entities = [e for e, _ in en_entity_counter.most_common(15)]
for en_e in top_en_entities:
    if en_e in en_to_zh_mapping:
        zh_mappings = en_to_zh_mapping[en_e].most_common(5)
        zh_str = ", ".join([f"{e}({c})" for e, c in zh_mappings])
        print(f"\n{en_e} -> {zh_str}")

## 5. Entity Category Analysis

Categorize entities into semantic groups (animals, body parts, nature, etc.)

In [ ]:
# Define entity categories (can be expanded)
CATEGORY_KEYWORDS = {
    'animals': {
        'zh': ['马', '牛', '羊', '鸡', '狗', '猪', '龙', '虎', '蛇', '兔', '鼠', '猴', '鸟', '鱼', '虫', '狼', '狐', '鹿', '象', '熊', '蜂', '蝶', '蚁', '鹰', '鸦', '雀', '燕', '鹤', '凤'],
        'en': ['dog', 'cat', 'horse', 'bird', 'fish', 'lion', 'tiger', 'bear', 'wolf', 'fox', 'sheep', 'cow', 'pig', 'chicken', 'duck', 'goose', 'snake', 'frog', 'bee', 'fly', 'ant', 'eagle', 'hawk', 'crow', 'dove', 'swan', 'mouse', 'rat', 'rabbit', 'deer', 'elephant', 'monkey', 'dragon']
    },
    'body_parts': {
        'zh': ['心', '眼', '手', '脚', '头', '口', '耳', '鼻', '舌', '牙', '骨', '血', '肉', '皮', '发', '面', '身', '腿', '臂', '指', '掌'],
        'en': ['heart', 'eye', 'hand', 'foot', 'head', 'mouth', 'ear', 'nose', 'tongue', 'tooth', 'bone', 'blood', 'flesh', 'skin', 'hair', 'face', 'body', 'leg', 'arm', 'finger', 'palm', 'brain', 'gut', 'neck', 'shoulder', 'back', 'chest']
    },
    'nature': {
        'zh': ['山', '水', '风', '雨', '云', '雪', '日', '月', '星', '天', '地', '火', '石', '木', '草', '花', '树', '河', '海', '雷', '电'],
        'en': ['mountain', 'water', 'wind', 'rain', 'cloud', 'snow', 'sun', 'moon', 'star', 'sky', 'earth', 'fire', 'stone', 'rock', 'wood', 'grass', 'flower', 'tree', 'river', 'sea', 'ocean', 'thunder', 'lightning', 'storm']
    },
    'human_relations': {
        'zh': ['人', '父', '母', '子', '女', '兄', '弟', '姐', '妹', '夫', '妻', '友', '敌', '王', '臣', '民'],
        'en': ['person', 'man', 'woman', 'father', 'mother', 'son', 'daughter', 'brother', 'sister', 'husband', 'wife', 'friend', 'enemy', 'king', 'queen', 'master', 'servant', 'child', 'baby']
    },
    'objects': {
        'zh': ['刀', '剑', '针', '线', '钱', '金', '银', '铁', '玉', '珠', '镜', '门', '窗', '床', '桌', '椅', '碗', '筷', '车', '船', '书'],
        'en': ['knife', 'sword', 'needle', 'thread', 'money', 'gold', 'silver', 'iron', 'stone', 'mirror', 'door', 'window', 'bed', 'table', 'chair', 'bowl', 'cup', 'car', 'boat', 'ship', 'book', 'pen', 'paper']
    },
    'food': {
        'zh': ['米', '饭', '面', '酒', '茶', '肉', '菜', '果', '豆', '油', '盐', '糖', '醋'],
        'en': ['bread', 'rice', 'meat', 'fish', 'fruit', 'wine', 'beer', 'tea', 'coffee', 'milk', 'egg', 'salt', 'sugar', 'honey', 'cake', 'pie', 'soup']
    }
}

def categorize_entity(entity, lang):
    """Categorize an entity based on keywords."""
    entity_lower = entity.lower()
    for category, keywords in CATEGORY_KEYWORDS.items():
        lang_keywords = keywords.get(lang, [])
        for kw in lang_keywords:
            if kw in entity_lower or entity_lower in kw:
                return category
    return 'other'

# Categorize all entities
zh_categories = Counter()
en_categories = Counter()

for entity, count in zh_entity_counter.items():
    cat = categorize_entity(entity, 'zh')
    zh_categories[cat] += count

for entity, count in en_entity_counter.items():
    cat = categorize_entity(entity, 'en')
    en_categories[cat] += count

print("Chinese Entity Categories:")
for cat, count in zh_categories.most_common():
    print(f"  {cat}: {count}")

print("\nEnglish Entity Categories:")
for cat, count in en_categories.most_common():
    print(f"  {cat}: {count}")

In [ ]:
# Visualize category distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Chinese categories
zh_cats = dict(zh_categories.most_common())
axes[0].pie(zh_cats.values(), labels=zh_cats.keys(), autopct='%1.1f%%', startangle=90)
axes[0].set_title('Chinese Entity Categories')

# English categories
en_cats = dict(en_categories.most_common())
axes[1].pie(en_cats.values(), labels=en_cats.keys(), autopct='%1.1f%%', startangle=90)
axes[1].set_title('English Entity Categories')

plt.tight_layout()
plt.show()

## 6. Interesting Examples: Same Meaning, Different Entities

In [ ]:
# Filter high-similarity pairs where both have entities and entities are different
interesting_pairs = both_df[
    (both_df['similarity'] >= 0.8) &  # High similarity
    (both_df['zh_entity_count'] > 0) &
    (both_df['en_entity_count'] > 0)
].sort_values('similarity', ascending=False)

print(f"Found {len(interesting_pairs)} high-similarity pairs with entities in both languages")
print("\n" + "=" * 80)
print("TOP 20 EXAMPLES: Same Meaning, Different Entities")
print("=" * 80)

for i, (_, row) in enumerate(interesting_pairs.head(20).iterrows(), 1):
    print(f"\n{i}. Similarity: {row['similarity']:.4f}")
    print(f"   Chinese: {row['zh_idiom']}")
    print(f"     Meaning: {row['zh_matched_meaning']}")
    print(f"     Entities: {row['zh_entities']}")
    print(f"   English: {row['en_idiom']}")
    print(f"     Meaning: {row['en_matched_meaning']}")
    print(f"     Entities: {row['en_entities']}")

## 7. Animal Entity Deep Dive

Animals are often used metaphorically with cultural differences.

In [ ]:
# Find pairs involving animal entities
zh_animals = set(CATEGORY_KEYWORDS['animals']['zh'])
en_animals = set(CATEGORY_KEYWORDS['animals']['en'])

def has_animal(entities, animal_set):
    for e in entities:
        for a in animal_set:
            if a in e.lower() or e.lower() in a:
                return True
    return False

animal_pairs = both_df[
    both_df.apply(lambda r: has_animal(r['zh_entities'], zh_animals) or has_animal(r['en_entities'], en_animals), axis=1)
]

print(f"Found {len(animal_pairs)} pairs involving animal entities")

# Show examples
print("\n" + "=" * 80)
print("ANIMAL-RELATED IDIOM PAIRS")
print("=" * 80)

for i, (_, row) in enumerate(animal_pairs.head(15).iterrows(), 1):
    print(f"\n{i}. Similarity: {row['similarity']:.4f}")
    print(f"   ZH: {row['zh_idiom']} (entities: {row['zh_entities']})")
    print(f"   EN: {row['en_idiom']} (entities: {row['en_entities']})")
    print(f"   Meaning: {row['en_matched_meaning']}")

## 8. Summary Statistics

In [ ]:
# Generate summary report
summary = {
    "Total cross-lingual pairs": len(df),
    "Pairs with Chinese entities": int(df['zh_has_entities'].sum()),
    "Pairs with English entities": int(df['en_has_entities'].sum()),
    "Pairs with BOTH having entities": int(df['both_have_entities'].sum()),
    "Unique Chinese entities": len(zh_entity_counter),
    "Unique English entities": len(en_entity_counter),
    "Average similarity score": round(df['similarity'].mean(), 4),
    "Median similarity score": round(df['similarity'].median(), 4),
    "Top Chinese entity": zh_entity_counter.most_common(1)[0] if zh_entity_counter else None,
    "Top English entity": en_entity_counter.most_common(1)[0] if en_entity_counter else None,
}

print("=" * 60)
print("SUMMARY REPORT")
print("=" * 60)
for key, value in summary.items():
    print(f"{key}: {value}")

In [ ]:
# Save summary to file
summary_output = {
    **summary,
    "zh_entity_frequency": dict(zh_entity_counter.most_common(100)),
    "en_entity_frequency": dict(en_entity_counter.most_common(100)),
    "zh_categories": dict(zh_categories),
    "en_categories": dict(en_categories),
    "top_entity_pairs": [(f"{zh}->{en}", count) for (zh, en), count in entity_pairs.most_common(50)]
}

with open('/home/jiaruil5/culture_pretrain/CultureInFigurativeLanguage/culture/data/idioms/entity_analysis_summary.json', 'w') as f:
    json.dump(summary_output, f, ensure_ascii=False, indent=2)

print("Summary saved to entity_analysis_summary.json")